# Week 5 — select, freeze and validate T5

**Objective:** choose a useful profile solution, keep its definitions fixed, and assess how it describes other countries and regions.

1. Select the worry coding, worry–harm allocation and number of profiles.
2. Check whether simple rules explain the profiles.
3. Verify the frozen assignments and assess held-out transfer.
4. Compare fixed profiles within regions with independently fitted regional groups.
5. Compare countries and record the limits of the results.

This is the short, current T5 workflow. The previous notebook, including the detailed search history, is preserved in `earlier iterations/before_minimal_T5_notebook_20261017`. Full tables and run protocols remain in `course_artifacts`; none are deleted.

**Run all cells to reproduce the assignments and country distances and read the saved experiment results. No clustering is fitted by default.** The final section identifies the original programs for reproducing the expensive searches and regional fits.

T5 uses complete responses and equal-country weights. There is no imputation, PCA or z-standardization. The 29 held-out countries had already been inspected before T5 selection before T5 was selected: T5 transfer is a secondary assessment, not an untouched external test.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy.spatial.distance import cdist, pdist, squareform
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits

# Open this notebook from the course folder or weekly_notebooks.
ROOT = Path.cwd()
if not (ROOT / 'course_artifacts').is_dir():
    ROOT = ROOT.parent
ART = ROOT / 'course_artifacts'
assert ART.is_dir(), 'Open the course folder first.'

# Fixed run folders: do not silently substitute a newer model or run.
SEARCH = ART / 'week04_kmeans_adaptive_optimization/54d309fe37735497'
REVIEW = ART / 'week04_kmeans_profile_evaluation/59d97cff396076a4'
TREES = ART / 'week04_t5_shallow_explanations/4f0f385fb4cbe644'
TRANSFER = ART / 'week05_t5_secondary_transfer/832b01713bc6b240'
FROZEN = TRANSFER / 'frozen_model'
REGIONAL = ART / 'week05_T5_completion/70ee2f67e95369c5'
COUNTRIES = REGIONAL / 'countries'

## 1. Choose the specification using development countries

Six worry answers are coded $0,1/t,1$. Thus $t$ changes the position of “somewhat worried”; the extremes remain 0 and 1. Write $q=1/t$. Allocation $\alpha$ controls the worry part of the squared-distance calculation.

For respondent $i$, let $m_i$ be the mean of the six coded worries. Use the six deviations from $m_i$ and $m_i$ itself, scaled by $\sqrt{\alpha/6}$. Scale the 14 personal/known-person harm indicators by $\sqrt{(1-\alpha)/14}$. The mean is derived from the six answers, not an additional independent question.

Every country has equal total weight. Within a country, preserve relative survey weights. The two short functions below define the common geometry and the ordinary weighted K-means fit. Defining the fit function does not run it.

In [ ]:
DOMAINS = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
WORRY = ['worry_' + domain for domain in DOMAINS]
PERSONAL = ['harm_' + domain + '_personal' for domain in DOMAINS + ['work']]
KNOWN = ['harm_' + domain + '_known_person' for domain in DOMAINS + ['work']]
FEATURES = WORRY + PERSONAL + KNOWN

def country_weights(frame):
    within_country = frame.WGT / frame.groupby('COUNTRY_ISO3').WGT.transform('sum')
    return (within_country / frame.COUNTRY_ISO3.nunique()).to_numpy()

def profile_matrix(frame, t, alpha):
    values = frame[FEATURES].to_numpy(float)
    assert np.isfinite(values).all(), 'Use complete responses only.'
    worry = np.where(values[:, :6] == 2, t, values[:, :6]) / t
    mean = worry.mean(axis=1, keepdims=True)
    # Equivalent to sqrt(alpha/6) and sqrt((1-alpha)/14).
    # Keep the original operation order for exact model replay.
    return np.column_stack([
        (worry - mean) * np.sqrt(alpha) / np.sqrt(6),
        mean * np.sqrt(alpha) / np.sqrt(6),
        values[:, 6:] * np.sqrt((1 - alpha) / 2) / np.sqrt(7),
    ])

def fit_kmeans(frame, t, alpha, k, seed):
    model = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=seed)
    with threadpool_limits(limits=1):
        model.fit(profile_matrix(frame, t, alpha), sample_weight=country_weights(frame))
    return model

### Search and stability criteria

Search $q=0.1$–$0.9$, $\alpha=0.2$–$0.8$, and $k=3,\ldots,10$: 301 initial settings, then 160 adaptively chosen settings. Ten repetitions per test screen the adaptive settings. Two settings per $k$, receive fresh confirmation with 50 repetitions per test.

Each repetition refits the complete clustering procedure on either 80% of respondents within each development country or 90 of the 113 development countries, stratified by region and income. Assign the common development respondents to the resulting centres. Fifty partitions give 1,225 pairwise adjusted Rand indices (ARIs); ARI measures agreement of membership, ignoring label names.

Retain a global candidate only if $k\geq3$, its largest equal-country share is at most 40%, dominance $D=k\max_j p_j<3$, and **both median and lower-decile ARI are at least 0.80 in both tests**. The lower decile is the 10th percentile. Pilot scores do not qualify a model. Pairwise ARIs are dependent summaries, not independent statistical observations.

The following table includes the best confirmed setting per $k$ including failures. “Retain” is recalculated from the stated rules.

In [ ]:
best = pd.read_csv(SEARCH / 'best_tested_setting_per_k.csv')
best['model'] = 'T' + best.k.astype(str)
comparison = best.sort_values('k').copy()
ari_columns = ['respondents_median', 'respondents_p10', 'countries_median', 'countries_p10']
comparison['retain'] = (
    comparison.k.ge(3) & comparison.largest.le(0.40) & comparison.dominance.lt(3)
    & comparison[ari_columns].ge(0.80).all(axis=1)
)
display(comparison[['model', 'k', 't', 'alpha', 'largest', 'dominance']
                   + ari_columns + ['retain']].round(4))

# A stable whole partition can still contain a poorly recovered profile.
recovery = pd.read_csv(REVIEW / 'model_review_summary.csv')
display(recovery[['model', 'k', 'respondents_min_cluster_jaccard_p10',
                  'countries_min_cluster_jaccard_p10']].round(3))

### Decision: use five profiles, T5

T5 has respondent ARI median/lower decile 0.974/0.946 and country values 0.956/0.913. Its largest share is 36.3%, giving dominance 1.816. Match repeated profiles one-to-one by weighted membership overlap; Jaccard is intersection divided by union. T5's weakest country-test profile has lower-decile Jaccard 0.943.

T3/T4 are stable but coarser. Finer solutions pass whole-partition ARI but have weaker individual profiles: weakest country lower-decile Jaccards are 0.419 (T6), 0.452 (T7), 0.471 (T8), 0.066 (T9) . T5 gives a useful moderate resolution, not a proven global optimum. There were no candidate-fitting failures; a seed-overlap preflight check was corrected before confirmation fitting.

Describe profiles in the original answer categories, not in transformed centre coordinates. The short names are summaries, not exact assignment rules.

In [ ]:
cards = pd.read_csv(REVIEW / 'profile_review_cards.csv')
display(cards.loc[cards.model.eq('T5'), [
    'cluster', 'description', 'weighted_share', 'respondent_share',
    'mean_worry_0_to_2', 'mean_personal_harm_prevalence',
    'mean_known_person_harm_prevalence']].round(3))

## 2. Check simple explanations

Reconstruct the fixed development labels; do not refit the clustering. Use all five country-grouped cross-validation folds and equal-country weights separately in training and evaluation.

- **Burden:** a decision tree using mean original worry and mean harm exposure.
- **Single answer:** for each profile versus everyone else, search the 20 original features, category thresholds and both directions in training countries. Select the best weighted balanced accuracy, then evaluate in the other countries. Use observed values only; no imputation.
- **Several answers:** `DecisionTreeClassifier(max_depth=depth, min_samples_leaf=200, random_state=2023)` on all 20 features, for depths 1, 2, 3 and 5. No class reweighting.

Balanced accuracy is mean recall over the target classes. Also inspect precision and recall for the single-answer checks. These are explanation checks, not independent clustering validation or prespecified rejection tests.

In [ ]:
for filename in ['burden_tree_summary.csv', 'single_feature_summary.csv',
                 'all_feature_tree_summary.csv']:
    print(filename)
    display(pd.read_csv(TREES / filename).round(4))

**Keep T5.** Burden-tree balanced accuracy at depths 1/2/3 is 32.7%/46.0%/51.2%. All-feature trees reach 36.8%/54.0%/67.9%; depth 5 reaches 73.7%. A five-class depth-1 tree cannot exceed 40% balanced accuracy; depth 2 cannot exceed 80%, because they have at most two and four leaves.

“Very worried about water” describes C4 well but does not define it exactly: 88.3% balanced accuracy, 67.4% precision and 84.3% recall. Neither the two burden variables nor single answers nearly reconstruct the complete five-profile solution.

## 3. Verify and apply the frozen model

The stored development centres, their order, the feature order and the eligibility rule are fixed. There is no held-out or all-country refit. Check the frozen file hashes, then load the 2023 prepared data. Worry is already coded 0/1/2; derive personal/known-person harm from the original four-category answers. Codes 98/99 remain missing.

Only complete responses on all 20 features receive a profile. Keep every other respondent in the ID registry with an exclusion reason; do not impute a label.

In [ ]:
seal = json.loads((FROZEN / 'freeze_seal.json').read_text())
for filename, expected in seal['files'].items():
    assert hashlib.sha256((FROZEN / filename).read_bytes()).hexdigest() == expected
specification = json.loads((FROZEN / 'specification.json').read_text())
centres = np.load(FROZEN / 'centres.npy')
t, alpha = specification['spec']['t'], specification['spec']['alpha']
assert specification['feature_order'] == FEATURES
print('Frozen T5: k = 5, t =', t, ', alpha =', alpha)

prepared = pd.read_parquet(ART / 'week03_2023_analysis_data.parquet')
sources = ['WP22442', 'WP22443', 'WP22444', 'WP22445', 'WP22446', 'WP22447', 'WP22448']
for personal, known, source in zip(PERSONAL, KNOWN, sources):
    raw = prepared[source]
    assert raw.dropna().isin([1, 2, 3, 4, 98, 99]).all()
    prepared[personal] = raw.map({1: 1., 2: 0., 3: 1., 4: 0.})
    prepared[known] = raw.map({1: 0., 2: 1., 3: 1., 4: 0.})
assert prepared[WORRY].stack().isin([0, 1, 2]).all()
assert prepared.WPID_RANDOM.is_unique
complete = prepared[FEATURES].notna().all(axis=1)
data = prepared.loc[complete].copy()

In [ ]:
# Nearest stored centre; argmin selects the first centre on an exact tie.
distances = cdist(profile_matrix(data, t, alpha), centres)
data['cluster'] = distances.argmin(axis=1)
nearest = np.sort(distances, axis=1)[:, :2]
data['distance'] = nearest[:, 0]
data['margin'] = np.divide(nearest[:, 1] - nearest[:, 0], nearest[:, 1],
                           out=np.zeros(len(data)), where=nearest[:, 1] > 0)
limits = np.array([specification['profile_distance_p95'][str(j)] for j in range(5)])
data['distance_flag'] = data.distance.to_numpy() > limits[data.cluster.to_numpy()]

# Compare by respondent ID, never by the order of rows in another file.
registry = pd.read_parquet(TRANSFER / 'all_respondents_T5_eligibility_and_frozen_profiles.parquet')
assert registry.WPID_RANDOM.is_unique
registered = registry.set_index('WPID_RANDOM')
assert set(prepared.WPID_RANDOM) == set(registered.index)
assert np.array_equal(complete, registered.loc[prepared.WPID_RANDOM, 'complete_case'])
expected = registered.loc[data.WPID_RANDOM]
assert np.array_equal(data.cluster, expected.T5_cluster)
assert np.allclose(data.distance, expected.T5_nearest_distance, atol=1e-12, rtol=0)
assert np.allclose(data.margin, expected.T5_margin, atol=1e-12, rtol=0)
assert np.array_equal(data.distance_flag, expected.T5_beyond_development_profile_p95)
print('All', len(data), 'frozen assignments reproduced; no model fitted.')
display(prepared.assign(assigned=complete).groupby('country_split').assigned.agg(['size', 'sum', 'mean']))

## 4. Assess held-out transfer

Compare the same fixed profiles in 113 development and 29 held-out countries. The complete-case counts are 109,488 and 27,972; 9,450 of the 146,910 prepared respondents remain unassigned.

Compare equal-country prevalence, original categories and the three fixed defining contrasts per profile. A frequency change alone is not poor transfer. A **distance flag** means the nearest-centre distance exceeds that profile's development weighted 95th percentile; it is not an error probability or an exclusion. The **margin** is (second-nearest distance − nearest distance) / second-nearest distance; a small margin indicates a less clear assignment.

Category differences use total variation: half the sum of absolute category-probability differences, averaged over features. The saved 1,000 country-bootstrap intervals hold the model fixed and resample the 29 held-out countries. Columns `p025`/`p975` are the 2.5th/97.5th percentiles for held-out profile shares, not prevalence changes. They do not cover worldwide sampling, missingness or model-selection uncertainty. No automatic transfer-rejection cutoff was specified.

In [ ]:
transfer = pd.read_csv(TRANSFER / 'profile_transfer_summary.csv')
display(transfer[['cluster', 'profile_name', 'equal_country_weighted_share_development',
                  'equal_country_weighted_share_heldout', 'prevalence_difference_pp',
                  'p025', 'p975', 'mean_feature_total_variation']].round(4))
anchors = pd.read_csv(TRANSFER / 'heldout_frozen_anchor_comparison.csv')
assert anchors.loc[anchors.scope.eq('global'), 'heldout_contrast_direction_preserved'].all()
display(pd.read_csv(TRANSFER / 'heldout_vs_development_quality.csv').round(4))
display(Image(filename=str(TRANSFER / 't5_original_response_transfer.png')))

country_quality = pd.read_csv(TRANSFER / 'heldout_country_quality_and_retention.csv')
display(country_quality.sort_values('profile_p95_exceedance', ascending=False)[
    ['Country', 'prepared', 'complete', 'retained_fraction', 'profile_p95_exceedance']].head(10).round(4))

**Conclusion:** all 15 defining contrasts retain their direction in aggregate. Mean feature-category differences are 0.008–0.028. The largest prevalence change is +2.1 percentage points for C3. Distance flags rise from 4.95% to 5.69%; median distance and margin change little. This supports aggregate descriptive transfer under fixed assignment, not independent rediscovery of five groups.

Do not hide country weaknesses: Sierra Leone has 18.8% flagged and 78.7% complete-case retention; Namibia 16.9% flagged; Somalia 16.7% flagged and 82.9% retention. Sweden's C4 cell has only 15 respondents (weighted effective size 9.8); Somalia's corresponding semantic shift has 173 respondents and is not the same tiny-cell issue. Full country/profile differences remain in `heldout_profile_semantic_differences.csv` in the transfer folder.

## 5. Compare geographical regions

**Two separate questions:**

- Apply the **same global T5 centres** everywhere: how do profile proportions and original answers vary by region?
- Fit groups **independently within each region**, using development countries only and T5's fixed coding/allocation: which local resolutions are stable?

The first comparison has a common meaning for C0–C4. The second does not: local labels must be compared by membership overlap and original answers, not by matching their numbers.

In [ ]:
# Average country proportions, not respondent counts pooled across countries.
data['country_weight'] = data.WGT / data.groupby('COUNTRY_ISO3').WGT.transform('sum')
shares = data.groupby(['COUNTRY_ISO3', 'cluster']).country_weight.sum().unstack(fill_value=0)
metadata = data.groupby('COUNTRY_ISO3')[['GlobalRegion', 'country_split']].first()
regional_share = shares.join(metadata).groupby('GlobalRegion')[list(range(5))].mean()
region_names = pd.read_csv(REGIONAL / 'development_regions.csv').set_index('GlobalRegion').region_name
regional_share.index = regional_share.index.map(region_names)
profile_names = {int(j): name for j, name in specification['profile_names'].items()}
display(regional_share.rename(columns=profile_names).round(3))

# Check the recalculated all-country regional proportions against the saved results.
saved = pd.read_csv(REGIONAL / 'T5_fixed_profiles_by_split_and_region.csv')
saved = saved.loc[saved.split.eq('all') & saved.area.ne('global')]
saved_share = saved.pivot(index='region_name', columns='cluster', values='equal_country_share')
assert np.allclose(regional_share, saved_share.loc[regional_share.index], atol=1e-12, rtol=0)
display(Image(filename=str(REGIONAL / 'figures/T5_region_prevalence.png')))

### Independent regional fits: use the existing protocol

Fit k=2–10 in each of the 15 regions, with 20 K-means initializations. Test k=5, k=2/3 and one inertia-based alternative per region (smallest eligible k within 95% of the maximum normalized chord–inertia gap). This is exploratory local selection, not independent confirmation.

Run 50 respondent tests retaining 80% within each country. With at least five development countries, run 50 country tests retaining floor(80% of countries); repeated country subsets are allowed and reported. With 2–4 countries, use leave-one-country-out diagnostics only; with one country, geographic stability is unavailable.

Require median **and** lower-decile pairwise ARI ≥0.80 in both full tests. Do not lower this threshold to retain more detailed groups. Report global size/dominance criteria as comparisons, **not universal regional filters**: a valid local binary division can be larger than 40%. Match local/global groups using weighted overlap and original category differences; Jaccard is descriptive, without an additional cutoff.

In [ ]:
local = pd.read_csv(REGIONAL / 'supported_local_resolutions.csv')
full_support = local.status.eq('both full stability benchmarks met')
display(local.loc[full_support, ['region_name', 'k', 'countries',
    'respondent_median_ari', 'respondent_p10_ari', 'country_median_ari',
    'country_p10_ari', 'largest_share', 'dominance']].round(3))
limited = local.status.ne('not qualified by the applicable stability benchmarks') & ~full_support
display(local.loc[limited, ['region_name', 'k', 'countries', 'status']])
display(Image(filename=str(REGIONAL / 'figures/T5_local_resolution_stability.png')))
display(pd.read_csv(REGIONAL / 'supported_local_semantic_correspondence.csv').round(3))

**Conclusion:** 135 regional reference fits and 5,228 stability fits succeeded, with no fit failures. Six alternatives pass both full tests: Eastern Africa k=3; Central/Western Africa k=3/4; Latin America/Caribbean k=3/4; Northern/Western Europe k=2. Northern Africa k=4 has limited leave-one-country-out support only. One-country development regions cannot establish geographic stability.

No k=5 solution passes both full tests in the 11 regions where those tests are available. This limits a universal interpretation of T5; it does not invalidate a common global description. Stable local groups need not be rich profiles: Northern/Western Europe's k=2 almost separates known-person mental-health harm (98.7% versus 0%).

Under fixed T5 assignment across all countries, lower-worry prevalence is 56.4% in Northern/Western Europe, 17.1% in Central/Western Africa and 13.8% in Southern Africa. Broad known-person harm is 8.2% versus 23.4% in the first two regions. These are equal-country complete-case summaries, not world-population prevalence. Original-category regional tables are saved alongside the prevalence tables.

## 6. Compare countries

Use the **common global profiles**, not unmatched local group labels. All-country comparisons are post-validation descriptions.

**Profile mixtures:** for country $a$, let $p_{aj}$ be its survey-weighted proportion in frozen T5 profile $j$. Compare countries $a,b$ by Hellinger distance:

$$H(a,b)=\sqrt{\frac12\sum_{j=0}^4(\sqrt{p_{aj}}-\sqrt{p_{bj}})^2}.$$

Each country summary needs **32 stored values**: 18 worry-category probabilities and 14 harm prevalences. For binary harm, $V_f(a,b)=|p_{af}-p_{bf}|$; the no-harm probability is redundant. The underlying variables and distances are unchanged.

**Original answers:** for each feature $f$, compare category probabilities by $V_f(a,b)=\frac12\sum_c|P_{afc}-P_{bfc}|$. Average within the six worries, seven personal harms and seven known-person harms, giving $V_W,V_S,V_K$. Combine as

$$V(a,b)=\alpha V_W(a,b)+\frac{1-\alpha}{2}\{V_S(a,b)+V_K(a,b)\}.$$

The original-category comparison uses T5's allocation but deliberately does not use numerical worry spacing $t$. It compares marginal distributions, not their within-person combinations. Recalculate both raw matrices below and check every value against the saved matrices.

In [ ]:
assert np.allclose(shares.sum(axis=1), 1)
iso = shares.index
hellinger = pd.DataFrame(squareform(pdist(np.sqrt(shares), metric='euclidean')) / np.sqrt(2),
                         index=iso, columns=iso)

# 32 country values: 18 worry-category probabilities and 14 harm prevalences.
feature_distances = []
for feature in FEATURES:
    if feature in WORRY:
        probabilities = data.groupby(['COUNTRY_ISO3', feature]).country_weight.sum().unstack(fill_value=0)
        probabilities = probabilities.reindex(index=iso, columns=[0, 1, 2], fill_value=0)
        assert np.allclose(probabilities.sum(axis=1), 1)
        distance = squareform(pdist(probabilities, metric='cityblock')) / 2
    else:
        # Binary total variation is the absolute gap in P(harm=1).
        prevalence = (data[feature] * data.country_weight).groupby(data.COUNTRY_ISO3).sum()
        prevalence = prevalence.reindex(iso).to_numpy()
        distance = np.abs(prevalence[:, None] - prevalence[None, :])
    feature_distances.append(distance)
worry_distance = np.mean(feature_distances[:6], axis=0)
personal_distance = np.mean(feature_distances[6:13], axis=0)
known_distance = np.mean(feature_distances[13:], axis=0)
response_distance = pd.DataFrame(
    alpha * worry_distance + (1 - alpha) * (personal_distance + known_distance) / 2,
    index=iso, columns=iso)

for metric, matrix in [('T5_profiles', hellinger), ('T5_responses', response_distance)]:
    saved = pd.read_csv(COUNTRIES / f'{metric}_all_countries_post_validation_raw_matrix.csv', index_col=0)
    assert np.allclose(matrix, saved.loc[iso, iso], atol=1e-12, rtol=0)
print('Both 142-country distance matrices reproduced.')

for country in ['USA', 'FIN']:
    print(country)
    display(pd.DataFrame({
        'profile-mixture nearest': hellinger.loc[country].drop(country).sort_values().head(5).index,
        'original-response nearest': response_distance.loc[country].drop(country).sort_values().head(5).index,
    }))

### Maps, geography, income and uncertainty

Map values divide raw distances by a **fixed all-country reference median**, never by a selected country's distances. Keep the previously fixed calibration values: 0.2218094853108838 for mixtures and 0.1550977480973838 for overall original responses. Worry/personal/known-person components have their own fixed medians. Value 1 means the reference median; values are uncapped, but the colour legend ends at “2 or more”. The scope and selected country never change the denominator. Approximate overview colours lose information; exact rankings use the raw matrices.

Saved extensions use 200 within-country respondent bootstraps with fixed labels and retained survey weights, plus 999 whole-country income-label permutations (also within regions). Pairs sharing a country are not independent. The respondent bootstraps are sensitivity checks, not full survey-design or model-selection uncertainty.

In [ ]:
for filename in ['country_geography_summary.csv', 'income_distance_summary.csv',
                 'income_country_label_permutations.csv', 'burden_only_information_loss.csv']:
    print(filename)
    display(pd.read_csv(COUNTRIES / filename).round(4))
uncertainty = pd.read_csv(COUNTRIES / 'conditional_nearest_neighbour_bootstrap.csv')
display(uncertainty.loc[uncertainty.scope.eq('all_countries_post_validation')
                        & uncertainty.iso3.isin(['USA', 'FIN'])].round(3))
display(Image(filename=str(COUNTRIES / 'T5_distance_from_USA_regions.png')))

**Interpretation:** same-region mean mixture distance is 0.174 versus 0.233 to other regions. Only 23.9% have their nearest country in the same region (random-peer reference 8.1%). An exploratory country-grouping cut gives two groups and region ARI 0.052; it is not adopted as a new regional classification.

Mixtures compress responses: Bosnia Herzegovina is nearest to the USA, with Iran third; Tajikistan is nearest to Finland. The original-answer comparison instead gives Canada and Iceland. Neither result measures societal equivalence, and familiar neighbours are not a reason to tune a distance. The exact nearest neighbour recurs in only 29.5% of conditional bootstraps for both USA and Finland (country median 40.5%). Read rankings as approximate proximity, not certain relationships.

Across 141 income-classified countries, income-gap rank correlation is 0.247 for mixtures and 0.298 for original answers. Within-region country-label permutations give upper-tail fractions 0.006/0.003; this is not causal evidence. Two burden totals cannot exactly recover T5: the within-sample best lookup reaches 60.1% accuracy in development and 58.2% held-out. That is descriptive information loss, not cross-validated performance or evidence of value for another outcome.

Interactive maps: open `github_publication/docs/index.html` for profiles, `countries.html` for country comparisons, or `country-income.html` for income. Full rankings, geography, income, burden, uncertainty and projection-quality results remain in the country folder.

## 7. Close Objective 1 and reuse the results

Use **frozen T5_C0–T5_C4** for the course continuation. Join the student mapping to the dataset by `WPID_RANDOM`, with a one-to-one check. Never join by row order, rename profiles by regional prevalence, impute excluded labels or refit the global model on all countries. Keep respondent-linked files off public GitHub.

The package `student_materials/T5_frozen_profiles` contains the mapping and instructions. Week 6 predicts these categorical labels using external variables; labels are targets, not predictors. T5 is a stable common global description with qualified transfer, not five independently established types in every region.

### Reproduction scope of this T5-only release

The notebook recalculates frozen assignments, regional prevalence and country distances. Saved tables report the completed parameter search, stability, explanatory checks and transfer. Historical alternative-model records and the full original search runners are not included in this public release; they remain in the private course archive. Published candidate tables omit those historical comparison rows. This is a documented subset, not a claim that no alternatives were tested.

`frozen_models/T5_models.json` and `public_assignment.py` allow fixed T5 assignment when survey data are supplied separately. The pages and their aggregate data work offline. `.week05_build/rebuild_T5_pages.py` refreshes the two map wrappers from the editable fragments. Full scientific replay requires the original data, private cache/sampling records and original software environment; this is not a zero-input reproduction bundle.